# 04 · Atacamos nuestros propios datos

Un dato sintético solo es anónimo si un adversario razonable **no** puede usarlo contra los
pacientes reales. Se ejecutan los ataques sobre la mitad de entrenamiento (miembros) y se comparan
con la mitad de control (no miembros): solo el exceso de acierto cuenta como fuga.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from synapse import datasets, generators, inference as inf, utility as ut, privacy as pv, compliance, viz
from synapse import experiment as ex
viz.style(); pd.set_option("display.precision", 3); pd.set_option("display.width", 140)
TABLES = ROOT / "results" / "tables"
metrics = pd.read_csv(TABLES / "study_metrics.csv") if (TABLES / "study_metrics.csv").exists() else None

In [2]:
d = datasets.load("gbsg2")
train, holdout = datasets.split_train_holdout(d.data, seed=0)
specs = {"bootstrap": {}, "marginal": {}, "copula": {}, "cart": {}, "dpcopula": {"epsilon": 1.0}}
res = {}
for g, p in specs.items():
    s = generators.make(g, seed=0, **p).fit(train, d.schema).sample(len(train))
    res[g] = pv.evaluate_privacy(train, holdout, s, d, seed=0)
pd.DataFrame(res).T[["exact_match_rate", "dcr_share_train", "mia_dist_auc", "mia_dist_tpr@10fpr", "mia_dens_auc",
                     "singling_out_risk", "linkability_risk", "inference_risk"]]

,exact_match_rate,dcr_share_train,mia_dist_auc,mia_dist_tpr@10fpr,mia_dens_auc,singling_out_risk,linkability_risk,inference_risk
bootstrap,1.0,1.000,0.808,0.661,0.598,0.754,0.596,0.590
marginal,0.0,0.494,0.485,0.102,0.539,0.162,0.000,0.000
copula,0.0,0.512,0.516,0.108,0.553,0.099,0.000,0.000
cart,0.0,0.672,0.594,0.216,0.561,0.773,0.097,0.083
dpcopula,0.0,0.358,0.506,0.082,0.514,0.000,0.000,0.035


**Comprobaciones de cordura.** El *bootstrap* (copiar filas) debe fallar todos los ataques y el
muestreo de marginales debe superarlos: si no fuera así, los ataques estarían mal construidos.

**Hallazgo.** CART al estilo `synthpop` tiene un riesgo de individualización tan alto como el
*bootstrap*: cada valor sintético es un valor real tomado de una hoja del árbol, así que las
combinaciones raras de valores reales sobreviven.

## Resultados del estudio completo

In [3]:
cols = ["exact_match_rate", "mia_dist_auc", "mia_dens_auc", "singling_out_risk", "linkability_risk", "inference_risk"]
metrics.groupby(["dataset", "label"])[cols].mean().round(3)

exact_match_rate  mia_dist_auc  mia_dens_auc  singling_out_risk  linkability_risk  inference_risk
dataset  label                                                                                                             
flchain  bootstrap                   1.000         0.813         0.600              0.752             0.628           0.656
         cart                        0.007         0.564         0.533              0.757             0.190           0.210
         copula                      0.000         0.503         0.508              0.048             0.013           0.055
         dpcopula(ε=0.5)             0.000         0.496         0.500              0.004             0.011           0.033
         dpcopula(ε=1)               0.000         0.498         0.506              0.004             0.014           0.057
         dpcopula(ε=10)              0.000         0.504         0.510              0.007             0.015           0.050
         dpcopula(ε=5)               0.000         0.499         0.509              0.005             0.017           0.045
         marginal                    0.000         0.503         0.506              0.049             0.019           0.038
gbsg2    bootstrap                   1.000         0.812         0.550              0.799             0.600           0.632
         cart                        0.001         0.581         0.530              0.798             0.167           0.159
         copula                      0.000         0.515         0.507              0.106             0.020           0.102
         dpcopula(ε=0.5)             0.000         0.504         0.503              0.005             0.004           0.054
         dpcopula(ε=1)               0.000         0.504         0.505              0.011             0.013           0.059
         dpcopula(ε=10)              0.000         0.507         0.515              0.014             0.013           0.054
         dpcopula(ε=5)               0.000         0.507         0.513              0.016             0.030           0.101
         marginal                    0.000         0.499         0.497              0.115             0.001           0.062
veterans bootstrap                   1.000         0.814         0.577              0.590             0.613           0.641
         cart                        0.006         0.634         0.555              0.317             0.317           0.266
         copula                      0.000         0.556         0.536              0.146             0.100           0.071
         dpcopula(ε=0.5)             0.000         0.538         0.487              0.013             0.120           0.133
         dpcopula(ε=1)               0.000         0.530         0.500              0.004             0.117           0.117
         dpcopula(ε=10)              0.000         0.539         0.509              0.009             0.076           0.139
         dpcopula(ε=5)               0.000         0.517         0.505              0.009             0.079           0.105
         marginal                    0.000         0.530         0.548              0.136             0.043           0.072
whas500  bootstrap                   1.000         0.823         0.539              0.767             0.553           0.654
         cart                        0.000         0.575         0.516              0.742             0.177           0.185
         copula                      0.000         0.529         0.525              0.076             0.037           0.089
         dpcopula(ε=0.5)             0.000         0.513         0.507              0.008             0.033           0.085
         dpcopula(ε=1)               0.000         0.510         0.507              0.016             0.034           0.069
         dpcopula(ε=10)              0.000         0.516         0.506              0.013             0.042           0.062
         dpcopula(ε=5)               0.000         0.520  

## Dos artefactos metodológicos detectados en el piloto

1. **DCR con muestras pequeñas.** En *veterans* (68 pacientes por mitad), la proporción de
   sintéticos más cercanos a entrenamiento que a control depende de la partición concreta y marcaba
   riesgo incluso con privacidad diferencial. Se informa, pero no puntúa en el semáforo.
2. **Secretos raros en Anonymeter.** Con `mgus` (1,5 % de «sí»), el riesgo de inferencia divide por
   un número casi nulo y el generador de marginales, que no puede filtrar nada, daba 0,24. Los
   secretos con clase minoritaria < 5 % se excluyen (`privacy.MIN_SECRET_PREVALENCE`).